# Deployment

## Objective

## 1. Load Model + Pipeline

In [1]:
import pandas as pd
import numpy as np
import joblib

best_model = joblib.load("../models/best_model_random_forest.pkl")

print(best_model)

RandomForestClassifier(random_state=42)


## 2. Load Encoder

In [3]:
encoder = joblib.load("../models/encoder.pkl")
print(encoder)

OneHotEncoder(handle_unknown='ignore', sparse_output=False)


## 3. Create Prediction Function
**Input**: raw dataframe (Sub_id1, Sub_id2, click_hour, click_day_of_week)   
**Output**: prediction (0 = Low, 1 = High)

In [19]:
def predict_commission_level(df):
    df = df.copy()

    # Input validation
    required = ['Sub_id1', 'Sub_id2', 'click_hour', 'click_day_of_week']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    cat_cols = ['Sub_id1', 'Sub_id2']
    
    # encode
    try:
        encoded = encoder.transform(df[cat_cols])
    except Exception as e:
        raise ValueError(f"Encoding failed: {e}")
    
    encoded_df = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(cat_cols),
        index=df.index
    )

    # drop + concat
    df = df.drop(columns=cat_cols)
    df = pd.concat([df, encoded_df], axis=1)

    # align columns ให้ตรงกับ model
    df = df.reindex(columns=best_model.feature_names_in_, fill_value=0)

    pred = best_model.predict(df)
    prob = best_model.predict_proba(df)[:, 1] 

    return pred, prob

## 4. Test with Sample Data

In [40]:
sample = pd.DataFrame({
    "Sub_id1": ["colgateOpticWhite"],
    "Sub_id2": ["purple"],
    "click_hour": [20],
    "click_day_of_week": [5]
})

prediction, probability = predict_commission_level(sample)
print("Prediction:", prediction)
print("Confidence:", probability)

Prediction: [1]
Confidence: [0.60416667]


## 5. Business Interpretation Layer

In [41]:
def interpret(pred, prob):
    return [
        f"HIGH COMMISSION → Recommended (confidence {p_conf:.2%})"
        if p == 1
        else f"LOW COMMISSION → Low priority (confidence {p_conf:.2%})"
        for p, p_conf in zip(pred, prob)
    ]
for text in interpret(prediction, probability):
    print(text)

HIGH COMMISSION → Recommended (confidence 60.42%)
